# Tutorial 02 · NumPy, Pandas, and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_02_NumPy_Pandas_and_Visualization/tutorial_02_data.ipynb)

Turn a small, intentionally messy table of synthetic building measurements into a clean dataset, summary tables, and interpretable plots.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python variables, lists, functions, and loops (Tutorial 01).

**Resources:** CPU only; no external dataset. NumPy, Pandas, Matplotlib, Seaborn, and Plotly install in the notebook if needed. Plan 45–60 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals

Understand array shape and broadcasting; distinguish element-wise from matrix operations; select and clean tabular data; group and join records; choose appropriate plots; and export your results. The analysis is descriptive, not a predictive model.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps the installed versions of NumPy, Pandas, Matplotlib, Seaborn, and Plotly when available. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'plotly': 'plotly'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import io, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
from IPython.display import display
%matplotlib inline

OUTPUT_DIR = Path("outputs/tutorial_02")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__, "seaborn": sns.__version__, "plotly": plotly.__version__}
display(pd.Series(versions, name="Versions"))

## 2. Arrays, shape, and random numbers

NumPy arrays usually store one dtype and support vectorized operations. `shape` tells you the axis lengths; `reshape` changes the arrangement without changing the element count. `arange` excludes the stop, whereas `linspace` includes it by default.

`rng.uniform` and `rng.normal` are different distributions. A standard normal sample can be negative and is not restricted to the interval [0, 1]. A seed reproduces a sequence within a compatible environment.

In [ ]:
vector = np.arange(6, dtype=np.float32)
matrix = vector.reshape(2, 3)
print("Vector:", vector, "shape:", vector.shape, "dtype:", vector.dtype)
print("Matrix:\n", matrix)
print("Five evenly spaced points:", np.linspace(0, 1, 5))
print("Zeros:\n", np.zeros((2, 3)), "\nIdentity:\n", np.eye(3))
uniform_draws = rng.uniform(0, 1, 1000)
normal_draws = rng.normal(0, 1, 1000)
assert ((uniform_draws >= 0) & (uniform_draws < 1)).all()
assert matrix.size == vector.size
print("Normal sample mean/std:", normal_draws.mean(), normal_draws.std())

## 3. Indexing, views, broadcasting, and matrix algebra

`matrix[row, column]` selects a value. Basic slices often share memory with their source; `.copy()` isolates changes. Boolean masks select observations matching a condition.

Broadcasting compares trailing dimensions: dimensions must match or one must equal one. A `[3]` vector can be added to each row of a `[2, 3]` matrix. `*` is element-wise multiplication; `@` is matrix multiplication. Use `np.linalg.solve(A, b)` to solve a system instead of explicitly computing an inverse.

In [ ]:
print("First row:", matrix[0], "Last column:", matrix[:, -1])
print("Values > 2:", matrix[matrix > 2])
independent = matrix[:, :2].copy()
independent[0, 0] = -99
assert matrix[0, 0] == 0
offset = np.array([10, 20, 30])
print("Broadcast addition:\n", matrix + offset)
print("Column means:", matrix.mean(axis=0), "Row means:", matrix.mean(axis=1))
A = np.array([[2., 1.], [1., 3.]])
b = np.array([5., 7.])
solution = np.linalg.solve(A, b)
np.testing.assert_allclose(A @ solution, b)
print("Element-wise square:\n", A * A, "\nMatrix square:\n", A @ A)
print("Solution:", solution, "Transpose:\n", A.T)
try:
    matrix + np.ones(2)
except ValueError as error:
    print("Expected incompatible-shape error:", error)

## 4. A DataFrame is a labeled table

The embedded CSV below is deliberately small and messy: it contains a duplicate, a missing value, inconsistent site labels, and a negative energy value. It represents **invented** daily records. No claim about real buildings follows from these numbers.

`loc` selects by labels or a mask; `iloc` selects by integer position. A Series is one labeled column. Keep units in column names when possible.

In [ ]:
csv_text = """date,site,temperature_c,energy_kwh
2026-01-01, A,5,24
2026-01-02,A,7,22
2026-01-03,A,9,20
2026-01-04,A,11,18
2026-01-05,A,13,16
2026-01-06,A,15,14
2026-01-01,B,5,31
2026-01-02,B,7,29
2026-01-03,B,9,
2026-01-04,B,11,25
2026-01-05,B,13,23
2026-01-06,B,15,-2
2026-01-02,A,7,22
"""
raw = pd.read_csv(io.StringIO(csv_text))
display(raw.head())
display(raw.dtypes)
print("Rows / columns:", raw.shape)
print("First temperature:", raw.iloc[0]["temperature_c"])
display(raw.loc[raw["temperature_c"] >= 11, ["site", "energy_kwh"]])

## 5. Clean transparently

Normalize labels, parse dates, remove exact duplicates, and mark invalid energy as missing. For this descriptive exercise we drop records missing required values and report every removal. Imputation can be useful, but it requires an explicit assumption. For machine learning, fit any imputation or scaling statistics on the training split only.

In [ ]:
clean = raw.copy()
clean["site"] = clean["site"].str.strip().str.upper()
clean["date"] = pd.to_datetime(clean["date"], errors="raise")
duplicates_removed = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()
invalid_energy = clean["energy_kwh"] < 0
invalid_count = int(invalid_energy.sum())
clean.loc[invalid_energy, "energy_kwh"] = np.nan
missing_removed = int(clean[["date", "site", "temperature_c", "energy_kwh"]].isna().any(axis=1).sum())
clean = clean.dropna(subset=["date", "site", "temperature_c", "energy_kwh"]).sort_values(["site", "date"]).reset_index(drop=True)
clean["energy_mj"] = clean["energy_kwh"] * 3.6
audit = {"raw_rows": len(raw), "duplicates_removed": duplicates_removed,
         "invalid_energy_marked_missing": invalid_count, "missing_rows_removed": missing_removed,
         "clean_rows": len(clean)}
assert audit == {"raw_rows":13, "duplicates_removed":1, "invalid_energy_marked_missing":1, "missing_rows_removed":2, "clean_rows":10}
assert (clean.energy_kwh >= 0).all() and not clean.duplicated(["date", "site"]).any()
print(audit)
display(clean)

## 6. Aggregate, pivot, and join

A mean and a total answer different questions. Group summaries should include counts, especially when missing data create unequal coverage. Joining site metadata uses a many-to-one check so duplicated metadata cannot silently multiply rows. A pivot aligns dates across sites; gaps remain visible rather than being invented.

In [ ]:
site_summary = clean.groupby("site").agg(days=("energy_kwh", "count"),
    mean_kwh=("energy_kwh", "mean"), total_kwh=("energy_kwh", "sum"))
site_metadata = pd.DataFrame({"site":["A","B"], "area_m2":[100.,150.]})
enriched = clean.merge(site_metadata, on="site", how="left", validate="many_to_one")
assert len(enriched) == len(clean) and enriched.area_m2.notna().all()
enriched["energy_kwh_per_m2"] = enriched.energy_kwh / enriched.area_m2
pivot = clean.pivot(index="date", columns="site", values="energy_kwh")
display(site_summary)
display(pivot)
print("Total recorded energy:", clean.energy_kwh.sum(), "kWh")
assert clean.energy_kwh.sum() == 222.0

## 7. Choose plots that answer a question

Use a line plot for time, a scatter plot for relationships, and a bar chart for group summaries. Always label units. Here the different totals partly reflect missing days, so the bar chart uses **daily means** and labels the number of observations. These hand-constructed data do not establish causal effects of temperature.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for site in pivot.columns:
    axes[0].plot(pivot.index, pivot[site], marker="o", label=site)
axes[0].set(xlabel="Date", ylabel="Daily energy (kWh)", title="Recorded time series")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Site")
sns.scatterplot(data=clean, x="temperature_c", y="energy_kwh", hue="site", ax=axes[1])
axes[1].set(xlabel="Temperature (°C)", ylabel="Daily energy (kWh)", title="Association, not causation")
bars = axes[2].bar(site_summary.index, site_summary.mean_kwh, color=["#2878a8", "#d28c31"])
axes[2].bar_label(bars, labels=[f"n={v}" for v in site_summary.days], padding=3)
axes[2].set(xlabel="Site", ylabel="Mean daily energy (kWh)", title="Unequal observation counts", ylim=(0,35))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "energy_overview.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(normal_draws, bins=30, alpha=0.65, label="Normal(0, 1)")
ax.hist(uniform_draws, bins=20, alpha=0.65, label="Uniform(0, 1)")
ax.set(xlabel="Draw", ylabel="Count", title="Different sampling distributions")
ax.legend()
plt.show()

## 8. Interactive visualization and export

Hover over points to inspect records. The HTML export includes Plotly’s JavaScript so the saved chart can be opened without fetching a separate script. CSV exports preserve the underlying values for reuse. A correlation matrix summarizes association; with tiny constructed data it is only a demonstration.

In [ ]:
interactive = px.scatter(enriched, x="temperature_c", y="energy_kwh", color="site",
    hover_data=["date", "energy_kwh_per_m2"], labels={"temperature_c":"Temperature (°C)", "energy_kwh":"Daily energy (kWh)"},
    title="Synthetic daily energy measurements")
interactive.show()
interactive.write_html(OUTPUT_DIR / "interactive_energy.html", include_plotlyjs=True)
clean.to_csv(OUTPUT_DIR / "clean_energy.csv", index=False)
site_summary.to_csv(OUTPUT_DIR / "site_summary.csv")
display(clean[["temperature_c", "energy_kwh"]].corr())
reloaded = pd.read_csv(OUTPUT_DIR / "clean_energy.csv", parse_dates=["date"])
pd.testing.assert_frame_equal(reloaded, clean, check_dtype=False)
(OUTPUT_DIR / "metrics.json").write_text(json.dumps({"versions":versions, "audit":audit,
    "total_energy_kwh":float(clean.energy_kwh.sum()), "synthetic":True}, indent=2))
print("TUTORIAL_02_COMPLETE — tables and figures saved to", OUTPUT_DIR)

## 9. Exercises

1. Predict the output shape of a `[4, 3]` array plus a `[3]` array and test your prediction.
2. Replace listwise deletion with a clearly labeled imputation rule; compare the group means and record the new assumptions.
3. Plot energy per square metre. Does the ranking change? What other differences would matter in a real comparison?
4. Build a Seaborn box plot and explain why ten observations give little information about a distribution’s tails.
5. Read the exported CSV in a fresh cell and reconstruct the daily mean table.

**Concept checks:** `axis=0` reduces across rows; `loc` is label-based; `*` and `@` are different operations; a missing record is not automatically zero; association does not establish causation.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_02/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

Adapted and reworked for independent Colab use from [PEESE Tutorial 02](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_02_NumPy_Pandas_and_Visualizations), Copyright (c) 2025 PEESE. The upstream MIT notice is retained in the repository `LICENSE`. This adaptation does not imply course endorsement.

References: [NumPy basics](https://numpy.org/doc/stable/user/basics.html), [Pandas tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/), [Matplotlib](https://matplotlib.org/stable/tutorials/index.html), [Seaborn](https://seaborn.pydata.org/tutorial.html), and [Plotly](https://plotly.com/python/). All measurements in this lesson are synthetic.